# 04 · Data and scoring

Validate record provenance and trial order, then compute reproducible trial-level metrics. The default input is a synthetic fixture set.

Run the setup cell first. See the repository README for ZIP and GitHub setup.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile
from zipfile import ZipFile

REPO_URL = ""
PROJECT_DIR = Path.cwd()

def find_project(directory):
    candidates = [directory, *sorted(path for path in directory.iterdir() if path.is_dir())]
    return next((path for path in candidates if (path / "src" / "pilot.py").is_file()), None)

project = find_project(PROJECT_DIR)
if project is None:
    if REPO_URL:
        project = Path(tempfile.mkdtemp(prefix="python-hci-")) / "repo"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(project)], check=True)
    else:
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("Extract the repository and set PROJECT_DIR to its root.") from exc
        import io
        uploaded = files.upload()
        archives = [data for name, data in uploaded.items() if name.lower().endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload one project ZIP.")
        directory = Path(tempfile.mkdtemp(prefix="python-hci-"))
        with ZipFile(io.BytesIO(archives[0])) as archive:
            archive.extractall(directory)
        project = find_project(directory)
if project is None or not (project / "study" / "stimuli.json").is_file():
    raise FileNotFoundError("Project root not found. Use the ZIP supplied with these notebooks.")
sys.path.insert(0, str(project.resolve()))
from src import pilot
OUT = project / "outputs"
print("Project root:", project)

For a single interactive browser walkthrough, replace the demo load with `records = [pilot.read_json(Path("path/to/download.json"))]` and run the scoring line. For actual participant data, set participant mode and point `RECORD_DIR` to the directory containing only your complete exports. An incomplete session should be documented separately. Keep all actual responses private.

In [2]:
MODE = "demo"
RECORD_DIR = project / "private_data"
if MODE == "demo":
    records = pilot.read_json(project / "demo" / "synthetic_records.json")
elif MODE == "participant":
    records = [pilot.read_json(path) for path in sorted(RECORD_DIR.glob("*.json"))]
    if not records:
        raise FileNotFoundError(f"No response JSON files in {RECORD_DIR}.")
else:
    raise ValueError("Choose demo or participant mode.")

scored = pilot.score_records(records, MODE)
print(f"{MODE.upper()} · {len(records)} records · {len(scored)} scored trials")

DEMO · 12 records · 36 scored trials


In [3]:
from collections import Counter

path = pilot.write_csv(OUT / MODE / "scored_trials.csv", scored)
print("Trials per condition:", dict(Counter(row["condition"] for row in scored)))
print("Scored trial CSV:", path)
print("Example row:", scored[0])

Trials per condition: {'A': 12, 'B': 12, 'C': 12}
Scored trial CSV: outputs/demo/scored_trials.csv
Example row: {'participant_id': 'P01', 'sequence': 1, 'position': 1, 'topic': 'lists', 'condition': 'A', 'rejected_inaccurate': 0, 'accepted_accurate': 1, 'initial_correct': 0, 'final_correct': 0, 'wrong_to_right': 0, 'right_to_wrong': 0, 'transfer_correct': 0, 'evidence_opened': '', 'source_link_opened': ''}


Correctness is binary for prediction, claim evaluation, and transfer. An “Unsure” response does not count as a correct answer. The change indicators capture initial-to-final revision in either direction. Evidence fields are blank for condition A, where the panel is unavailable. The CSV omits free text. See the session guide for manual coding of evidence use.